# Test based on https://github.com/NathanWycoff/prox_alasso/blob/main/python/ada_sim.py

## Function to generate data

In [32]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV
from adaptivelasso.skcompat import MAPAdaptiveLasso
from scipy.special import expit


def generate_data(lik, N, P, beta=None, sigma=1.0, random_state=None):
    rng = np.random.default_rng(random_state)
    X = rng.normal(size=(N, P))
    if beta is None:
        beta = np.zeros(P)
        beta[0] = 1.0  # Ensure y depends on X[:,0]
    mu = X @ beta

    if lik == 'normal':
        y = mu + sigma * rng.normal(size=N)
    elif lik == 'cauchy':
        y = mu + sigma * rng.standard_cauchy(size=N)
    elif lik == 'bernoulli':
        logits = mu
        probs = expit(logits)
        y = rng.binomial(1, probs)
    elif lik == 'poisson':
        lam = np.exp(mu)
        y = rng.poisson(lam)
    elif lik == 'zinb':
        zero_prob = 0.3
        is_zero = rng.uniform(size=N) < zero_prob
        mu_nb = np.exp(mu)
        y_nb = rng.negative_binomial(n=1/sigma**2, p=1/(1+mu_nb*sigma**2), size=N)
        y = np.where(is_zero, 0, y_nb)
    else:
        raise ValueError(f"Unknown likelihood: {lik}")
    return X, y

## Report model results

In [37]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def report_model_results(model, X_test, y_test, lik):
    # Estimated coefficients
    if hasattr(model, 'coef_'):
        est_coefs = model.coef_
    elif hasattr(model, 'get_coefficients'):
        est_coefs = model.get_coefficients()
    else:
        est_coefs = None
    print("Estimated coefficients:", est_coefs)

    # Predictions
    y_pred = model.predict(X_test)

    # R² and RMSE
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"R²: {r2:.4f}")
    print(f"RMSE: {rmse:.4f}")

    # Distribution-specific parameters
    if lik == 'normal':
        error = y_test - y_pred
        print("Estimated error variance:", np.var(error))
    elif lik == 'cauchy':
        error = y_test - y_pred
        print("Estimated error MAD:", np.median(np.abs(error - np.median(error))))
    elif lik == 'zinb':
        percent_zeros = np.mean(y_pred == 0)
        print("Percent zeros in predictions:", percent_zeros)

## Compare MAPAdaptiveLasso with LassoCV

In [38]:
# Example for normal likelihood
N, P = 1000, 20
beta = np.zeros(P)
beta[:5] = 2  # 5 informative features
sigma = 1.0
lik = 'bernoulli'

X, y = generate_data(lik, N, P, beta, sigma=sigma, random_state=42)

# Split and fit as before...
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=5000, track=False))
])
pipe.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe.named_steps['jax_model'], X_test, y_test, lik)

# Use a pipeline for scaling and LassoCV
pipe_lassocv = Pipeline([
    ('scale', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=0, max_iter=5000))
])
pipe_lassocv.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe_lassocv.named_steps['lasso'], X_test, y_test, lik=lik)

Estimated coefficients: [ 1.43269653  1.33293229  1.27725445  1.39471951  1.3978965   0.02935551
  0.19004085  0.16904123  0.02996322 -0.         -0.          0.10877891
  0.05453544  0.11963845 -0.          0.00685595  0.0241327   0.02171606
  0.15254156  0.03193648]
R²: 0.5703
RMSE: 0.3254
Estimated coefficients: [ 0.16488544  0.1550696   0.14862804  0.16592588  0.16575047 -0.
  0.01683911  0.01516307  0.0040984  -0.0040288  -0.          0.01580763
 -0.          0.00102194  0.         -0.00028995  0.00109467  0.00600582
  0.01435396  0.        ]
R²: 0.4902
RMSE: 0.3544


In [48]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from adaptivelasso.skcompat import MAPAdaptiveLasso

# Assume generate_data and report_model_results are already defined as above

likelihoods = ['normal', 'cauchy', 'bernoulli', 'poisson', 'zinb']
results = []
max_iters = 3000  # Set a maximum number of iterations

for lik in likelihoods:
    print(f"Testing likelihood: {lik}")
    N, P = 1000, 20
    true_coefs = np.zeros(P)
    true_coefs[:3] = 1.0
    sigma = 1.0

    X, y = generate_data(lik, N, P, beta=true_coefs, sigma=sigma, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

    # MAPAdaptiveLasso
    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=max_iters, track=False))
    ])
    pipe.fit(X_train, y_train)
    y_pred_adapt = pipe.predict(X_test)
    r2_adapt = r2_score(y_test, y_pred_adapt)
    rmse_adapt = np.sqrt(mean_squared_error(y_test, y_pred_adapt))
    coefs_adapt = pipe.named_steps['jax_model'].get_coefficients()
    coefs_adapt_rounded = np.round(coefs_adapt, 3).tolist()

    # Always fit LassoCV, regardless of likelihood
    pipe_lasso = Pipeline([
        ('scale', StandardScaler()),
        ('lasso', LassoCV(cv=5, random_state=0, max_iter=max_iters))
    ])
    pipe_lasso.fit(X_train, y_train)
    y_pred_lasso = pipe_lasso.predict(X_test)
    if lik == 'bernoulli':
        y_pred_lasso_bin = (y_pred_lasso > 0.5).astype(int)
        r2_lasso = r2_score(y_test, y_pred_lasso_bin)
        rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso_bin))
    else:
        r2_lasso = r2_score(y_test, y_pred_lasso)
        rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
    coefs_lasso = pipe_lasso.named_steps['lasso'].coef_
    coefs_lasso_rounded = np.round(coefs_lasso, 3).tolist()

    better_r2 = 'AdaptiveLasso' if r2_adapt > r2_lasso else 'LassoCV'
    better_rmse = 'AdaptiveLasso' if rmse_adapt < rmse_lasso else 'LassoCV'

    results.append({
        'likelihood': lik,
        'R2_AdaptiveLasso': r2_adapt,
        'RMSE_AdaptiveLasso': rmse_adapt,
        'Coefs_AdaptiveLasso': coefs_adapt_rounded[:5],  # Show first 5 for brevity
        'R2_LassoCV': r2_lasso,
        'RMSE_LassoCV': rmse_lasso,
        'Coefs_LassoCV': coefs_lasso_rounded[:5],        # Show first 5 for brevity
        'Better_R2': better_r2,
        'Better_RMSE': better_rmse
    })

df_results = pd.DataFrame(results)
print(df_results)

Testing likelihood: normal
ES stop!
Testing likelihood: cauchy
Testing likelihood: bernoulli
Testing likelihood: poisson
ES stop!
Testing likelihood: zinb
  likelihood  R2_AdaptiveLasso  RMSE_AdaptiveLasso  \
0     normal          0.730321            0.947799   
1     cauchy          0.020204            7.062426   
2  bernoulli          0.192847            0.448399   
3    poisson          0.965784            2.473868   
4       zinb          0.395319           11.591741   

                     Coefs_AdaptiveLasso  R2_LassoCV  RMSE_LassoCV  \
0  [1.007, 1.042, 1.041, -0.028, -0.014]    0.736476      0.936920   
1   [1.009, 1.014, 0.903, 0.111, -0.077]    0.000429      7.133340   
2    [0.937, 0.939, 0.936, 0.007, 0.005]   -0.304697      0.570088   
3  [0.985, 0.985, 0.975, -0.021, -0.006]    0.355186     10.739441   
4  [0.808, 0.792, 1.029, -0.148, -0.019]    0.176886     13.524318   

                         Coefs_LassoCV      Better_R2    Better_RMSE  
0     [0.96, 0.997, 1.009, -